# Layer Composition Analysis

Analyzes how layers contribute to the model's output and interact. Measures per-layer importance, redundancy, similarity, and specialization.

**References:**
- Veit et al. (2016) "Residual Networks Behave Like Ensembles"
- Men et al. (2024) "ShortGPT: Layers in LLMs are More Redundant Than You Think"

## Why This Matters

Layer composition analysis measures how each layer's contribution interacts with previous layers — do they add independent information, reinforce existing predictions, or interfere? Understanding composition patterns (additive, multiplicative, corrective) reveals the model's computational strategy.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.layer_composition import (
    layer_residual_contribution,
    layer_output_similarity,
    layer_redundancy_analysis,
    critical_layer_identification,
    layer_specialization_profile,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35])
def metric(logits): return float(logits[-1, 0])
print(f"Model: {cfg.n_layers} layers")

In [ ]:
# 1. Residual contribution per layer
contrib = layer_residual_contribution(model, tokens)
for l in range(cfg.n_layers):
    print(f"Layer {l}: contribution={contrib['layer_contributions'][l]:.4f}, "
          f"fraction={contrib['contribution_fractions'][l]:.3f}")
print(f"Dominant layer: {contrib['dominant_layer']}")

In [ ]:
# 2. Layer output similarity
sim = layer_output_similarity(model, tokens)
print(f"Consecutive similarity: {[f'{s:.3f}' for s in sim['consecutive_similarity']]}")
print(f"Most different layer: {sim['most_different_layer']}")

In [ ]:
# 3. Layer redundancy
red = layer_redundancy_analysis(model, tokens, metric)
print(f"Most redundant: layer {red['most_redundant_layer']}")
print(f"Most critical: layer {red['most_critical_layer']}")

In [ ]:
# 4. Specialization profile
spec = layer_specialization_profile(model, tokens)
for l in range(cfg.n_layers):
    print(f"Layer {l}: attn={spec['attn_norms'][l]:.3f}, mlp={spec['mlp_norms'][l]:.3f}, "
          f"attn_frac={spec['attn_fraction'][l]:.3f}")